In [ ]:
pip install nltk scikit-learn pandas numpy PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 6.3 MB/s eta 0:00:00


In [ ]:
import PyPDF2
import nltk
import re
import pandas as pd
import numpy as np
from google.colab import files

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

nltk.download('punkt')
nltk.download('stopwords')

from nltk.corpus import stopwords

def upload_resume():
    uploaded = files.upload()
    if not uploaded:
        return None
    return list(uploaded.keys())[0]

def extract_text_from_pdf(file_path):
    text = ""
    with open(file_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            if page.extract_text():
                text += page.extract_text()
    return text.lower()


def preprocess(text):
    text = re.sub(r'[^a-zA-Z ]', '', text)
    tokens = nltk.word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    tokens = [w for w in tokens if w not in stop_words]
    return " ".join(tokens)

skills_db = [
    "python", "java", "machine learning", "deep learning",
    "nlp", "data science", "sql", "html", "css",
    "javascript", "react", "node", "c++",
    "excel", "communication", "leadership", "management",
    "marketing", "sales", "finance", "accounting",
    "business analysis", "business intelligence",
    "project management", "agile", "strategy"
]

def extract_skills(text):
    return [skill for skill in skills_db if skill in text]

jobs = [
    # 💻 TECH ROLES
    {"title": "Software Engineer", "description": "java c++ python algorithms data structures system design"},
    {"title": "Web Developer", "description": "html css javascript react node frontend backend"},
    {"title": "Frontend Developer", "description": "html css javascript react ui ux design"},
    {"title": "Backend Developer", "description": "node python java api database server"},
    {"title": "Full Stack Developer", "description": "frontend backend react node database api"},
    {"title": "Mobile App Developer", "description": "android ios flutter react native mobile apps"},
    {"title": "DevOps Engineer", "description": "docker kubernetes ci cd cloud aws automation"},
    {"title": "Cloud Engineer", "description": "aws azure cloud computing infrastructure deployment"},
    {"title": "Cybersecurity Analyst", "description": "network security ethical hacking penetration testing"},
    {"title": "Database Administrator", "description": "sql database management optimization backup"},

    # 📊 DATA & AI ROLES
    {"title": "Data Scientist", "description": "python machine learning data science statistics nlp"},
    {"title": "Data Analyst", "description": "excel sql data visualization business insights"},
    {"title": "AI Engineer", "description": "deep learning neural networks python nlp"},
    {"title": "Machine Learning Engineer", "description": "machine learning python models deployment"},
    {"title": "Business Intelligence Analyst", "description": "data visualization dashboards power bi tableau"},

    # 💼 BUSINESS ROLES
    {"title": "Business Analyst", "description": "business analysis data analysis communication sql reporting"},
    {"title": "Product Manager", "description": "product strategy roadmap agile stakeholder management"},
    {"title": "Operations Manager", "description": "operations management process optimization leadership"},
    {"title": "Project Manager", "description": "project planning agile scrum management leadership"},
    {"title": "Entrepreneur", "description": "business strategy innovation startup leadership"},

    # 💰 FINANCE ROLES
    {"title": "Financial Analyst", "description": "finance accounting forecasting excel financial modeling"},
    {"title": "Accountant", "description": "accounting bookkeeping taxation financial records"},
    {"title": "Investment Analyst", "description": "investment portfolio stock market analysis finance"},
    {"title": "Banking Officer", "description": "banking operations customer service finance"},

    # 📢 MARKETING ROLES
    {"title": "Marketing Manager", "description": "digital marketing seo branding analytics communication"},
    {"title": "SEO Specialist", "description": "seo search engine optimization keywords analytics"},
    {"title": "Content Writer", "description": "content writing blogging communication creativity"},
    {"title": "Social Media Manager", "description": "social media marketing branding engagement content"},
    {"title": "Sales Manager", "description": "sales negotiation communication business development"},

    # 🏥 HEALTHCARE ROLES
    {"title": "Doctor", "description": "medical diagnosis patient care healthcare treatment"},
    {"title": "Nurse", "description": "patient care medical assistance healthcare"},
    {"title": "Pharmacist", "description": "medicine drugs prescription healthcare"},
    {"title": "Medical Lab Technician", "description": "lab testing medical analysis diagnostics"},

    # 🎨 CREATIVE ROLES
    {"title": "Graphic Designer", "description": "design photoshop illustrator creativity branding"},
    {"title": "UI UX Designer", "description": "ui ux design user experience prototyping"},
    {"title": "Video Editor", "description": "video editing premiere pro creativity media"},
    {"title": "Animator", "description": "animation 3d design creativity graphics"},

    # ⚙️ GENERAL / OTHER ROLES
    {"title": "HR Manager", "description": "recruitment employee management leadership communication"},
    {"title": "Teacher", "description": "teaching education subject knowledge communication"},
    {"title": "Lawyer", "description": "law legal advice court case documentation"},
    {"title": "Civil Engineer", "description": "construction design infrastructure engineering"},
    {"title": "Mechanical Engineer", "description": "machines design manufacturing engineering"},
    {"title": "Electrical Engineer", "description": "circuits power systems electrical engineering"}
]

job_df = pd.DataFrame(jobs)

def match_jobs(resume_text, job_df):
    documents = [resume_text] + job_df['description'].tolist()

    tfidf = TfidfVectorizer(ngram_range=(1,2))
    vectors = tfidf.fit_transform(documents)

    similarity = cosine_similarity(vectors[0:1], vectors[1:]).flatten()

    job_df['similarity'] = similarity
    return job_df.sort_values(by='similarity', ascending=False)

def train_models():
    X = np.array([
        [3, 0.2], [5, 0.5], [2, 0.1], [7, 0.8], [6, 0.7]
    ])
    y = np.array([50, 70, 40, 90, 80])

    lr = LinearRegression()
    dt = DecisionTreeRegressor()
    rf = RandomForestRegressor()

    lr.fit(X, y)
    dt.fit(X, y)
    rf.fit(X, y)

    return lr, dt, rf

def predict_score(models, features):
    lr, dt, rf = models

    lr_pred = lr.predict([features])[0]
    dt_pred = dt.predict([features])[0]
    rf_pred = rf.predict([features])[0]

    final_score = (lr_pred + dt_pred + rf_pred) / 3

    return lr_pred, dt_pred, rf_pred, final_score

def calculate_job_chance(skill_count, similarity):
    skill_score = min(skill_count / 10, 1)
    similarity_score = similarity

    chance = (0.6 * similarity_score + 0.4 * skill_score) * 100
    return chance

def suggest_improvements(user_skills, job_description):
    job_words = job_description.split()
    missing = []

    for word in job_words:
        if word not in user_skills:
            missing.append(word)

    return list(set(missing))

def main():
    print("📂 Upload your resume...")
    resume_path = upload_resume()

    if not resume_path:
        print("❌ No file selected")
        return

    print("✅ File selected:", resume_path)

    text = extract_text_from_pdf(resume_path)
    clean_text = preprocess(text)

    skills = extract_skills(clean_text)
    print("\n🧠 Extracted Skills:", skills)

    matched_jobs = match_jobs(clean_text, job_df)

    print("\n🎯 Top Job Matches:")
    print(matched_jobs[['title', 'similarity']].head(5))

    best_job = matched_jobs.iloc[0]
    best_similarity = best_job['similarity']
    best_description = best_job['description']

    skill_count = len(skills)

    models = train_models()
    lr, dt, rf, final_score = predict_score(models, [skill_count, best_similarity])

    print("\n📊 Resume Score Prediction:")
    print(f"Linear Regression: {lr:.2f}")
    print(f"Decision Tree: {dt:.2f}")
    print(f"Random Forest: {rf:.2f}")
    print(f"🔥 Final Score: {final_score:.2f}")

    job_chance = calculate_job_chance(skill_count, best_similarity)
    print(f"\n🎯 Chance of Getting Job: {job_chance:.2f}%")

    if job_chance >= 75:
        print("✅ High chances of getting this job")
    elif job_chance >= 50:
        print("⚠️ Moderate chances – improve some skills")
    else:
        print("❌ Low chances – need significant improvement")

    # Improvements
    missing_skills = suggest_improvements(skills, best_description)

    print("\n📉 Missing Skills to Improve:")
    print(missing_skills)

# Run
if __name__ == "__main__":
    main()

📂 Upload your resume...


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Saving Joshi shreena resume .pdf to Joshi shreena resume  (1).pdf
✅ File selected: Joshi shreena resume  (1).pdf

🧠 Extracted Skills: ['excel', 'leadership', 'management', 'finance']

🎯 Top Job Matches:
                            title  similarity
14  Business Intelligence Analyst    0.097957
21                     Accountant    0.055299
11                   Data Analyst    0.041663
20              Financial Analyst    0.041294
17             Operations Manager    0.039842

📊 Resume Score Prediction:
Linear Regression: 60.00
Decision Tree: 40.00
Random Forest: 49.50
🔥 Final Score: 49.83

🎯 Chance of Getting Job: 21.88%
❌ Low chances – need significant improvement

📉 Missing Skills to Improve:
['visualization', 'tableau', 'data', 'dashboards', 'power', 'bi']
